# 02 · Limpieza, transformación y EDA

Diagnóstico de calidad sobre la capa raw, tratamiento de nulos, corrección de importes, verificación de integridad y exportación de la capa clean (informe técnico, apartados 4.1 a 4.3 y 4.5).


### 1. Carga y Exploración Inicial de los Datasets

#### 1.1 Estructura general

| Dataset | Filas | Columnas |
|---|---|---|
| Clientes | 1.000 | 6 |
| Transacciones | 8.000 | 7 |
| Productos | 48 | 4 |
| Tiendas | 5 | 3 |

#### 1.2 Nulos detectados por dataset

**Clientes**

| Campo | Nulos | % sobre total |
|---|---|---|
| customer_id | 0 | 0% |
| nombre | 0 | 0% |
| email | 40 | 4% |
| ciudad | 50 | 5% |
| canal_adquisicion | 0 | 0% |
| fecha_alta | 0 | 0% |

**Transacciones**

| Campo | Nulos | % sobre total |
|---|---|---|
| transaction_id | 0 | 0% |
| customer_id | 0 | 0% |
| product_id | 0 | 0% |
| store_id | 240 | 3% |
| cantidad | 0 | 0% |
| importe | 0 | 0% |
| fecha | 0 | 0% |

**Productos**

| Campo | Nulos | % sobre total |
|---|---|---|
| product_id | 0 | 0% |
| nombre | 0 | 0% |
| categoria | 0 | 0% |
| precio_base | 0 | 0% |

**Tiendas**

| Campo | Nulos | % sobre total |
|---|---|---|
| store_id | 0 | 0% |
| nombre | 0 | 0% |
| ciudad | 0 | 0% |

#### 1.3 Validación contra suciedad intencional documentada

| Campo | Nulos esperados | Nulos reales | Validación |
|---|---|---|---|
| clientes.email | 4% (40) | 40 | ✅ |
| clientes.ciudad | 5% (50) | 50 | ✅ |
| transacciones.store_id | 3% (240) | 240 | ✅ |

> ✅ Los tres tipos de suciedad intencional documentada se generaron correctamente y coinciden con los porcentajes definidos en el diseño del dataset.

In [1]:
import pandas as pd
import numpy as np

import os

RAW_PATH   = os.path.join('..', '..', 'data', 'raw')
CLEAN_PATH = os.path.join('..', '..', 'data', 'clean')

# Cargar los 4 datasets
clientes = pd.read_csv(os.path.join(RAW_PATH, 'clientes.csv'))
transacciones = pd.read_csv(os.path.join(RAW_PATH, 'transacciones.csv'))
productos = pd.read_csv(os.path.join(RAW_PATH, 'productos.csv'))
tiendas = pd.read_csv(os.path.join(RAW_PATH, 'tiendas.csv'))

# Exploración inicial
for nombre, df in [('CLIENTES', clientes), ('TRANSACCIONES', transacciones), 
                   ('PRODUCTOS', productos), ('TIENDAS', tiendas)]:
    print(f"\n{'='*40}")
    print(f"📂 {nombre}")
    print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
    print(f"Nulos:\n{df.isnull().sum()}")


📂 CLIENTES
Filas: 1000 | Columnas: 6
Nulos:
customer_id           0
nombre                0
email                40
ciudad               50
canal_adquisicion     0
fecha_alta            0
dtype: int64

📂 TRANSACCIONES
Filas: 8000 | Columnas: 7
Nulos:
transaction_id      0
customer_id         0
product_id          0
store_id          240
cantidad            0
importe             0
fecha               0
dtype: int64

📂 PRODUCTOS
Filas: 48 | Columnas: 4
Nulos:
product_id     0
nombre         0
categoria      0
precio_base    0
dtype: int64

📂 TIENDAS
Filas: 5 | Columnas: 3
Nulos:
store_id    0
nombre      0
ciudad      0
dtype: int64


### 2. Análisis exploratorio y limpieza de nulos en cada dataset

#### Tratamiento de valores nulos — dataset clientes

**Contexto:** antes de cualquier imputación o eliminación, se realizó un análisis
exploratorio para determinar si los nulos tenían un origen sistémico o eran ruido aleatorio.

##### Metodología aplicada

Se analizaron 3 dimensiones de nulidad sobre las columnas `email` (4%) y `ciudad` (5%):

- **Solapamiento:** verificar si los mismos clientes concentraban ambos nulos
- **Canal de adquisición:** detectar si algún canal fallaba sistemáticamente en la captura
- **Temporalidad:** identificar períodos con anomalías en la calidad del dato

##### Hallazgos

- Solo 1 cliente de 1.000 presentó ambos campos nulos simultáneamente
- La tasa de nulos por canal oscila entre 2.8% y 6.1% sin outliers significativos
- No se identificó ningún período con concentración anómala de nulos (ene 2023 – dic 2025)

##### Decisión y justificación

Se decidió **conservar los nulos sin imputación ni eliminación de filas**, por dos razones:

1. Ninguna de las dos columnas participa en las preguntas analíticas del proyecto
(segmentación RFM, LTV, canal de venta, categoría de producto, estacionalidad)
2. El análisis confirmó que los nulos son aleatorios, independientes entre sí
y sin patrón de origen — no hay problema sistémico que corregir

Eliminar las filas habría significado descartar información válida de 89 clientes
(transacciones, canal, fecha de alta) para resolver un problema inexistente.

##### 2.1.1 Patrones de nulidad -> solapamiento : ¿ los mismos clientes tienen ambos nulos?

In [2]:
null_email  = clientes['email'].isna()
null_ciudad = clientes['ciudad'].isna()

solo_email  = null_email  & ~null_ciudad
solo_ciudad = ~null_email & null_ciudad
ambos       = null_email  & null_ciudad

print(f"Solo email nulo:   {solo_email.sum()}")
print(f"Solo ciudad nulo:  {solo_ciudad.sum()}")
print(f"Ambos nulos:       {ambos.sum()}")
print(f"\nClientes con algún nulo: {(null_email | null_ciudad).sum()}")

Solo email nulo:   39
Solo ciudad nulo:  49
Ambos nulos:       1

Clientes con algún nulo: 89


##### 2.1.2 Patrones de nulidad -> canal — ¿los nulos se concentran en algún canal?

In [3]:
# Bloque 2 — canal
print("=== email nulo por canal ===")
print(clientes[null_email]['canal_adquisicion'].value_counts())

print("\n=== ciudad nulo por canal ===")
print(clientes[null_ciudad]['canal_adquisicion'].value_counts())

print("\n=== % de nulos por canal ===")
tasa = clientes.groupby('canal_adquisicion').agg(
    total=('customer_id', 'count'),
    nulo_email=('email',  lambda x: x.isna().sum()),
    nulo_ciudad=('ciudad', lambda x: x.isna().sum())
)
tasa['pct_email']  = (tasa['nulo_email']  / tasa['total'] * 100).round(1)
tasa['pct_ciudad'] = (tasa['nulo_ciudad'] / tasa['total'] * 100).round(1)
print(tasa[['total','pct_email','pct_ciudad']])

=== email nulo por canal ===
canal_adquisicion
paid        13
organico    11
tienda       9
referral     7
Name: count, dtype: int64

=== ciudad nulo por canal ===
canal_adquisicion
referral    15
organico    14
paid        12
tienda       9
Name: count, dtype: int64

=== % de nulos por canal ===
                   total  pct_email  pct_ciudad
canal_adquisicion                              
organico             229        4.8         6.1
paid                 233        5.6         5.2
referral             249        2.8         6.0
tienda               289        3.1         3.1


##### 2.1.3 Patrones de nulidad -> temporalidad — ¿los nulos aparecen en un período concreto?

In [4]:
# Bloque 3 — temporalidad
clientes['fecha_alta'] = pd.to_datetime(clientes['fecha_alta'])
clientes['anio_mes']   = clientes['fecha_alta'].dt.to_period('M')

evolucion = clientes.groupby('anio_mes').agg(
    total=('customer_id', 'count'),
    nulo_email=('email',  lambda x: x.isna().sum()),
    nulo_ciudad=('ciudad', lambda x: x.isna().sum())
)
evolucion['pct_email']  = (evolucion['nulo_email']  / evolucion['total'] * 100).round(1)
evolucion['pct_ciudad'] = (evolucion['nulo_ciudad'] / evolucion['total'] * 100).round(1)

print(evolucion[evolucion['nulo_email'] + evolucion['nulo_ciudad'] > 0])

          total  nulo_email  nulo_ciudad  pct_email  pct_ciudad
anio_mes                                                       
2023-01      28           1            3        3.6        10.7
2023-02      22           2            3        9.1        13.6
2023-03      27           2            0        7.4         0.0
2023-04      34           0            2        0.0         5.9
2023-05      35           0            3        0.0         8.6
2023-06      34           2            1        5.9         2.9
2023-07      31           3            0        9.7         0.0
2023-08      28           0            3        0.0        10.7
2023-09      16           1            1        6.2         6.2
2023-10      29           0            3        0.0        10.3
2023-11      32           1            4        3.1        12.5
2023-12      32           2            2        6.2         6.2
2024-01      34           1            1        2.9         2.9
2024-02      22           0            1

#### 2.2 Tratamiento de valores nulos - dataset transacciones

**Campo afectado:** `store_id` — 240 nulos (3% de 8.000 transacciones)

#### Análisis previo a la decisión

Se analizaron 3 dimensiones antes de imputar:

- **Distribución temporal:** los nulos aparecen de forma uniforme entre enero 2023
y diciembre 2025, entre 3 y 11 registros por mes, sin concentración en ningún período
- **Ticket medio:** 220.35 — coherente con el rango del resto de tiendas (228–240)
y con T05 · Cumbres Web (236.30) específicamente
- **Catálogo de tiendas:** existe explícitamente una tienda `T05 · Cumbres Web · Online`,
lo que indica que el canal web está contemplado en el sistema

#### Hipótesis

Las transacciones con `store_id` nulo corresponden a ventas web que no registraron
correctamente su identificador de tienda. La distribución uniforme a lo largo de 3 años
descarta un bug puntual y apunta a un problema crónico de configuración en el origen.

#### Limitaciones

> **Nota:** esta es una asunción basada en evidencia circunstancial, no en validación
> externa. En un entorno productivo se requeriría confirmación del equipo técnico
> o cruce con logs del sistema antes de imputar.
> En este proyecto los datos son simulados, por lo que se acepta la hipótesis
> y se documenta explícitamente.

####  Decisión 

Se imputan los 240 nulos con el valor `T05`, ya que `store_id` es la clave del desglose por punto de venta
(pregunta analítica 4 del informe: *¿cómo varía el comportamiento de compra entre puntos de venta y categorías de producto?*)
y la evidencia disponible respalda que corresponden a ventas online. El efecto de esta imputación sobre la lectura del canal online se analiza en el apartado 7.4 del informe.
```python
transacciones['store_id'] = transacciones['store_id'].fillna('T05')
```

---

*Herramientas: Python · pandas*

In [5]:
print(tiendas)

  store_id             nombre     ciudad
0      T01     Cumbres Madrid     Madrid
1      T02  Cumbres Barcelona  Barcelona
2      T03   Cumbres Valencia   Valencia
3      T04    Cumbres Sevilla    Sevilla
4      T05        Cumbres Web     Online


In [6]:
# ¿Cuántas transacciones tiene T05 vs nulos?
print(transacciones['store_id'].value_counts(dropna=False))

store_id
T04    1622
T05    1591
T02    1531
T01    1525
T03    1491
NaN     240
Name: count, dtype: int64


In [7]:
comparacion = transacciones.groupby(
    transacciones['store_id'].fillna('NULO')
)['importe'].agg(['mean','count']).round(2)
print(comparacion)

            mean  count
store_id               
NULO      220.35    240
T01       228.26   1525
T02       230.03   1531
T03       240.14   1491
T04       233.91   1622
T05       236.30   1591


In [8]:
# ¿Los nulos se concentran en algún período concreto?
transacciones['fecha'] = pd.to_datetime(transacciones['fecha'])
transacciones['anio_mes'] = transacciones['fecha'].dt.to_period('M')

temporal = transacciones[transacciones['store_id'].isna()].groupby('anio_mes').size()
print(temporal)

anio_mes
2023-01    10
2023-02     4
2023-03     9
2023-04     7
2023-05    11
2023-06     4
2023-07     5
2023-08     5
2023-09     3
2023-10     6
2023-11     6
2023-12     5
2024-01     3
2024-02     6
2024-03     7
2024-04    10
2024-05     7
2024-06     6
2024-07     4
2024-08     9
2024-09     6
2024-10     8
2024-11     9
2024-12     8
2025-01     7
2025-02     5
2025-03     9
2025-04     9
2025-05     6
2025-06     7
2025-07     6
2025-08     7
2025-09     8
2025-10     5
2025-11     5
2025-12     8
Freq: M, dtype: int64


In [9]:
## Se agregaron los 240 nulos en la columna 'T05'
transacciones['store_id'] = transacciones['store_id'].fillna('T05')

In [10]:
print(transacciones['store_id'].value_counts(dropna=False))

store_id
T05    1831
T04    1622
T02    1531
T01    1525
T03    1491
Name: count, dtype: int64


### 3. Estadística descriptiva

##### 3.1 Variables numéricas

Análisis y tratamiento de las variables numéricas de `transacciones` (`importe`,
`cantidad`). Se detectan dos señales en `importe`, se corrige la que corresponde a
un error de origen y se documenta la que es comportamiento legítimo del negocio.

**Diagnóstico inicial (`importe`)**

- **Sesgo a la derecha.** Media (233) por encima de la mediana (210), con un máximo
  (3.198) muy despegado del P75 (308): una cola de importes altos tira de la media.
- **Valores negativos.** Mínimo −2.378,40. La magnitud es demasiado alta para un
  ticket del negocio, lo que obliga a verificar origen y patrón antes de interpretarlos.

**Tratamiento — errores POS (×8)**

La cola alta no son transacciones de alto valor: son errores de registro POS con
mecanismo de inyección conocido (×8). El importe legítimo vive en [20, 400]; los
errores se desplazaron a [160, 3.200]. El P99 (1.583) solo capturaba la mitad, por lo
que el umbral correcto es el dominio válido (400), no un percentil. Se corrigen 146
registros dividiendo por 8, lo que restaura el importe original y conserva el signo.
Tras la corrección, ningún importe supera el dominio válido (máx 399,79).

**Negativos — devoluciones**

Los 80 registros negativos (1%) se distribuyen en [−398, −21], uniformes entre tiendas
(0,76%–1,21%). Son devoluciones legítimas del ciclo de venta y se conservan: excluirlas
daría una imagen artificialmente positiva del valor del cliente y distorsionaría el LTV.

**`cantidad`** — limpia. Rango 1–3, sin valores imposibles.

In [11]:
# Estadísticas descriptivas — variables numéricas
print(transacciones[['importe', 'cantidad']].describe().round(2))

       importe  cantidad
count  8000.00   8000.00
mean    233.32      1.39
std     271.86      0.63
min   -2378.40      1.00
25%     111.36      1.00
50%     210.32      1.00
75%     308.83      2.00
max    3198.16      3.00


In [12]:
# ── Corrección de errores POS (×8) ─────────────────────────
# El importe legítimo se generó en [20, 400]. Los errores se inyectaron ×8 → [160, 3200].
# El P99 (1.583) solo capturó ~la mitad; el resto quedó escondido en [400, 1.583].
# Umbral correcto = dominio válido conocido (400), no un percentil.

# Cuantificar antes de corregir
err = transacciones[transacciones['importe'].abs() > 400]
rev_bruto = transacciones.loc[transacciones['importe'] > 0, 'importe'].sum()
print(f"Errores POS detectados (|importe|>400): {len(err)}")
print(f"% sobre facturación bruta: {err.loc[err['importe']>0,'importe'].sum() / rev_bruto:.1%}")

# Corregir por el mecanismo conocido: ÷8 restaura el importe original y conserva el signo
transacciones.loc[transacciones['importe'].abs() > 400, 'importe'] /= 8

# Verificar
print("Máx importe:", round(transacciones['importe'].max(), 2))   # ~400
print("Mín importe:", round(transacciones['importe'].min(), 2))   # ~-400
print("Negativos:",   (transacciones['importe'] < 0).sum())       # ~80 devoluciones

Errores POS detectados (|importe|>400): 146
% sobre facturación bruta: 14.0%
Máx importe: 399.79
Mín importe: -398.17
Negativos: 80


In [13]:
## Se observa qué sucede con los valores negativos: 

negativos = transacciones[transacciones['importe'] < 0]
print(f"Total negativos: {len(negativos)}")
print(f"% sobre total: {round(len(negativos)/len(transacciones)*100, 2)}%")
print(negativos[['importe', 'store_id', 'cantidad']].describe().round(2))

Total negativos: 80
% sobre total: 1.0%
       importe  cantidad
count    80.00     80.00
mean   -199.76      1.39
std     110.17      0.58
min    -398.17      1.00
25%    -283.58      1.00
50%    -195.76      1.00
75%    -112.95      2.00
max     -21.04      3.00


In [14]:
print(negativos['store_id'].value_counts())
print()
print(round(negativos['store_id'].value_counts() / 
      transacciones['store_id'].value_counts() * 100, 2))

store_id
T01    18
T03    18
T04    15
T02    15
T05    14
Name: count, dtype: int64

store_id
T01    1.18
T02    0.98
T03    1.21
T04    0.92
T05    0.76
Name: count, dtype: float64


In [15]:
print(transacciones[['importe']].quantile([0.90, 0.95, 0.99, 1.0]).round(2))

      importe
0.90   361.44
0.95   380.73
0.99   396.43
1.00   399.79


In [16]:
# Verificación post-corrección: no debe quedar NINGÚN registro sobre el dominio válido
errores_residuales = transacciones[transacciones['importe'].abs() > 400]
print(f"Registros sobre umbral 400 tras corrección: {len(errores_residuales)}")  # → 0
assert (transacciones['importe'].abs() <= 400).all(), "Quedan errores POS sin corregir"
print("✅ Todos los importes dentro del dominio válido [−400, 400]")

Registros sobre umbral 400 tras corrección: 0
✅ Todos los importes dentro del dominio válido [−400, 400]


#### 4. Verificar consistencia entre tablas

##### Consistencia entre tablas — integridad referencial

Se verificó que todos los identificadores presentes en `transacciones` existen
en sus tablas de referencia correspondientes.

##### Conclusión

La integridad referencial está garantizada. Todos los joins entre `transacciones`
y las tablas `clientes`, `productos` y `tiendas` devolverán resultados completos
sin pérdida de registros.


In [17]:
# customer_id
clientes_en_trans = set(transacciones['customer_id'].unique())
clientes_en_tabla = set(clientes['customer_id'].unique())
inconsistentes_clientes = clientes_en_trans - clientes_en_tabla

# product_id
productos_en_trans = set(transacciones['product_id'].unique())
productos_en_tabla = set(productos['product_id'].unique())
inconsistentes_productos = productos_en_trans - productos_en_tabla

# store_id
tiendas_en_trans = set(transacciones['store_id'].unique())
tiendas_en_tabla = set(tiendas['store_id'].unique())
inconsistentes_tiendas = tiendas_en_trans - tiendas_en_tabla

print(f"customer_id inconsistentes: {len(inconsistentes_clientes)}")
print(f"product_id inconsistentes:  {len(inconsistentes_productos)}")
print(f"store_id inconsistentes:    {len(inconsistentes_tiendas)}")

customer_id inconsistentes: 0
product_id inconsistentes:  0
store_id inconsistentes:    0


#### 5. Distribución temporal

###### Período analizado
Enero 2023 – diciembre 2025 (36 meses continuos, sin gaps).

###### Hallazgos

Volumen de transacciones Oscila entre 178 y 255 transacciones por mes, manteniéndose estable alrededor de la media durante todo el período. No se identificaron meses sin actividad ni picos anómalos significativos.

Importe total mensual — Se mueve entre ~35.700 y ~52.400 €, consistente con el volumen estable de transacciones. Sin tendencia creciente ni decreciente a lo largo de los 3 años.

Ticket medio — Oscila entre ~187 y ~220 €, con variabilidad natural mes a mes pero sin patrón estacional ni tendencia reconocible.

###### Conclusión
El dataset es temporalmente limpio y consistente. La ausencia de estacionalidad y tendencia es coherente con el origen sintético de los datos y no afecta la validez del análisis posterior.


In [18]:
transacciones['fecha'] = pd.to_datetime(transacciones['fecha'])
transacciones['anio_mes'] = transacciones['fecha'].dt.to_period('M')

temporal = transacciones.groupby('anio_mes').agg(
    transacciones=('transaction_id', 'count'),
    importe_total=('importe', 'sum'),
    ticket_medio=('importe', 'mean')
).round(2)

print(temporal)

          transacciones  importe_total  ticket_medio
anio_mes                                            
2023-01             230       44562.04        193.75
2023-02             217       45849.23        211.29
2023-03             234       46641.94        199.32
2023-04             206       42784.06        207.69
2023-05             212       42155.50        198.85
2023-06             215       45046.67        209.52
2023-07             222       45313.20        204.11
2023-08             233       43579.85        187.04
2023-09             205       42302.25        206.35
2023-10             255       52369.97        205.37
2023-11             219       46315.37        211.49
2023-12             213       44517.67        209.00
2024-01             197       39347.67        199.73
2024-02             225       45337.49        201.50
2024-03             228       48311.95        211.89
2024-04             227       47853.21        210.81
2024-05             235       47380.95        

#### 6. Distribución variables categóricas: dataset transacciones

##### Por tienda

| Tienda | Transacciones | % | Ticket medio | Importe total |
|---|---|---|---|---|
| T01 · Madrid | 1.525 | 19,1% | 205,56 | 313.478 |
| T02 · Barcelona | 1.531 | 19,1% | 205,64 | 314.837 |
| T03 · Valencia | 1.491 | 18,6% | 207,73 | 309.723 |
| T04 · Sevilla | 1.622 | 20,3% | 202,47 | 328.399 |
| T05 · Web | 1.831 | 22,9% | 204,19 | 373.872 |

T05 · Web lidera en volumen (22,9%) y facturación (373,9k €, ~22,8% del total). El
ticket medio es homogéneo entre canales (202–208 €): la diferenciación de valor viene
de la frecuencia de compra, no del gasto por visita — coherente con el RFM.

##### Por categoría

| Categoría | Transacciones | % | Ticket medio | Importe total |
|---|---|---|---|---|
| Ciclismo | 2.036 | 25,4% | 203,20 | 413.711 |
| Montaña | 2.022 | 25,3% | 206,03 | 416.599 |
| Padel | 2.013 | 25,2% | 204,74 | 412.135 |
| Running | 1.929 | 24,1% | 206,25 | 397.864 |

Distribución equilibrada en volumen y ticket medio homogéneo (~205 €): ninguna
categoría destaca en gasto por visita.

##### Por producto

Los 10 productos más frecuentes oscilan entre 2,2% y 2,5%, sin ninguno dominante.
Catálogo balanceado entre los 48 productos.

##### Conclusión

Distribución uniforme entre tiendas, categorías y productos. Sin concentración que
sesgue el análisis posterior.


In [19]:
# Distribución por tienda
print("=== transacciones por tienda ===")
tienda_dist = transacciones.groupby('store_id').agg(
    transacciones=('transaction_id', 'count'),
    importe_total=('importe', 'sum'),
    ticket_medio=('importe', 'mean')
).round(2)
tienda_dist['pct'] = (tienda_dist['transacciones'] / len(transacciones) * 100).round(1)
print(tienda_dist)

# Distribución por producto
print("\n=== transacciones por producto ===")
producto_dist = transacciones.groupby('product_id').agg(
    transacciones=('transaction_id', 'count'),
    importe_total=('importe', 'sum')
).round(2)
producto_dist['pct'] = (producto_dist['transacciones'] / len(transacciones) * 100).round(1)
print(producto_dist.sort_values('transacciones', ascending=False).head(10))

=== transacciones por tienda ===


          transacciones  importe_total  ticket_medio   pct
store_id                                                  
T01                1525      313477.52        205.56  19.1
T02                1531      314836.91        205.64  19.1
T03                1491      309723.39        207.73  18.6
T04                1622      328398.66        202.47  20.3
T05                1831      373871.91        204.19  22.9

=== transacciones por producto ===
            transacciones  importe_total  pct
product_id                                   
P037                  197       39655.95  2.5
P025                  193       39757.39  2.4
P009                  188       39022.07  2.4
P014                  183       38011.16  2.3
P017                  183       37551.65  2.3
P048                  180       39386.90  2.2
P006                  178       37105.73  2.2
P013                  178       34351.82  2.2
P032                  178       37150.65  2.2
P027                  177       36308.45  2.2

In [20]:
# Cruzar transacciones con categoría de producto
trans_cat = transacciones.merge(
    productos[['product_id', 'categoria']], 
    on='product_id', 
    how='left'
)

print("=== transacciones por categoría ===")
cat_dist = trans_cat.groupby('categoria').agg(
    transacciones=('transaction_id', 'count'),
    importe_total=('importe', 'sum'),
    ticket_medio=('importe', 'mean')
).round(2)
cat_dist['pct'] = (cat_dist['transacciones'] / len(transacciones) * 100).round(1)
print(cat_dist.sort_values('importe_total', ascending=False))

=== transacciones por categoría ===
           transacciones  importe_total  ticket_medio   pct
categoria                                                  
Montaña             2022      416598.67        206.03  25.3
Ciclismo            2036      413710.85        203.20  25.4
Padel               2013      412134.94        204.74  25.2
Running             1929      397863.93        206.25  24.1


#### 7. Duplicados por dataset

In [21]:
# Duplicados por dataset
for nombre, df in [('clientes', clientes), ('transacciones', transacciones),
                   ('productos', productos), ('tiendas', tiendas)]:
    duplicados = df.duplicated().sum()
    print(f"{nombre}: {duplicados} duplicados")

clientes: 0 duplicados


transacciones: 0 duplicados
productos: 0 duplicados
tiendas: 0 duplicados


#### 8. Transformación de los tipos de datos en formato de lectura MySQL

In [22]:
for nombre, df in [('clientes', clientes), ('transacciones', transacciones),
                   ('productos', productos), ('tiendas', tiendas)]:
    print(f"\n=== {nombre} ===")
    print(df.dtypes)


=== clientes ===
customer_id                     str
nombre                          str
email                           str
ciudad                          str
canal_adquisicion               str
fecha_alta           datetime64[us]
anio_mes                  period[M]
dtype: object

=== transacciones ===
transaction_id               str
customer_id                  str
product_id                   str
store_id                     str
cantidad                   int64
importe                  float64
fecha             datetime64[us]
anio_mes               period[M]
dtype: object

=== productos ===
product_id         str
nombre             str
categoria          str
precio_base    float64
dtype: object

=== tiendas ===
store_id    str
nombre      str
ciudad      str
dtype: object


In [23]:
# clientes
clientes['fecha_alta'] = pd.to_datetime(clientes['fecha_alta']).dt.strftime('%Y-%m-%d')

# transacciones
transacciones['fecha'] = transacciones['fecha'].dt.strftime('%Y-%m-%d')
transacciones = transacciones.drop(columns=['anio_mes'])

# Verificar
for nombre, df in [('clientes', clientes), ('transacciones', transacciones),
                   ('productos', productos), ('tiendas', tiendas)]:
    print(f"\n=== {nombre} ===")
    print(df.dtypes)


=== clientes ===
customer_id                str
nombre                     str
email                      str
ciudad                     str
canal_adquisicion          str
fecha_alta                 str
anio_mes             period[M]
dtype: object

=== transacciones ===
transaction_id        str
customer_id           str
product_id            str
store_id              str
cantidad            int64
importe           float64
fecha                 str
dtype: object

=== productos ===
product_id         str
nombre             str
categoria          str
precio_base    float64
dtype: object

=== tiendas ===
store_id    str
nombre      str
ciudad      str
dtype: object


#### 9. Exportación de datos CLEAN

In [24]:
clientes.to_csv(os.path.join(CLEAN_PATH, 'clientes_clean.csv'), index=False, encoding='utf-8')
transacciones.to_csv(os.path.join(CLEAN_PATH, 'transacciones_clean.csv'), index=False, encoding='utf-8')
productos.to_csv(os.path.join(CLEAN_PATH, 'productos_clean.csv'), index=False, encoding='utf-8')
tiendas.to_csv(os.path.join(CLEAN_PATH, 'tiendas_clean.csv'), index=False, encoding='utf-8')

print("Archivos exportados correctamente")

Archivos exportados correctamente
